In [1]:
!pip install xlstm transformers -q

In [3]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import ndcg_score

from xlstm import (
    xLSTMBlockStack,
    xLSTMBlockStackConfig,
    mLSTMBlockConfig,
    mLSTMLayerConfig,
    FeedForwardConfig,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

Device: cuda


In [4]:
metrics_df = pd.read_csv("captions_metrics.csv")
print(metrics_df.shape)

unique_images = metrics_df["image_idx"].unique()
rng = np.random.RandomState(SEED)
rng.shuffle(unique_images)

n = len(unique_images)
n_train = int(n * 0.8)
n_val = int(n * 0.1)

train_ids = set(unique_images[:n_train])
val_ids = set(unique_images[n_train:n_train + n_val])
test_ids = set(unique_images[n_train + n_val:])

train_df = metrics_df[metrics_df["image_idx"].isin(train_ids)].reset_index(drop=True)
val_df = metrics_df[metrics_df["image_idx"].isin(val_ids)].reset_index(drop=True)
test_df = metrics_df[metrics_df["image_idx"].isin(test_ids)].reset_index(drop=True)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

(413915, 8)
Train: 331130, Val: 41390, Test: 41395


In [5]:
def evaluate_selection(df, score_col, target_col="cider", group_col="image_idx", k=3):
    top1_ciders = []
    is_best_flags = []
    ndcgs = []

    for _, group in df.groupby(group_col):
        group = group.sort_values("caption_number")
        true_relevance = group[target_col].values.reshape(1, -1)
        pred_scores = group[score_col].values.reshape(1, -1)

        selected_idx = np.argmax(group[score_col].values)
        selected_row = group.iloc[selected_idx]
        top1_ciders.append(selected_row[target_col])
        is_best_flags.append(bool(selected_row["is_best_cider"]))

        if true_relevance.shape[1] > 1 and true_relevance.std() > 1e-8:
            ndcgs.append(ndcg_score(true_relevance, pred_scores, k=min(k, true_relevance.shape[1])))

    return {
        "avg_cider_selected": float(np.mean(top1_ciders)),
        "top1_accuracy": float(np.mean(is_best_flags)),
        f"ndcg@{k}": float(np.mean(ndcgs)) if ndcgs else None,
    }

In [6]:
TOKENIZER_NAME = "bert-base-uncased"
MAX_LEN = 32

tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
VOCAB_SIZE = tokenizer.vocab_size
print("Vocab size:", VOCAB_SIZE)

Vocab size: 30522


In [7]:
class CaptionScoreDatasetXLSTM(Dataset):
    def __init__(self, df, tokenizer, max_len=32, target_col="cider"):
        self.texts = df["caption_text"].astype(str).tolist()
        self.targets = df[target_col].astype(float).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        item["target"] = torch.tensor(self.targets[idx], dtype=torch.float)
        return item

train_dataset = CaptionScoreDatasetXLSTM(train_df, tokenizer, MAX_LEN)
val_dataset = CaptionScoreDatasetXLSTM(val_df, tokenizer, MAX_LEN)
test_dataset = CaptionScoreDatasetXLSTM(test_df, tokenizer, MAX_LEN)

BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [9]:
EMBED_DIM = 256  # сильно меньше hidden у RoBERTa/Mamba — модель обучается с нуля,
                  # большой размер при отсутствии претрейна привёл бы к недообучению
                  # на относительно небольшом датасете (330K коротких подписей)

xlstm_cfg = xLSTMBlockStackConfig(
    mlstm_block=mLSTMBlockConfig(
        mlstm=mLSTMLayerConfig(
            conv1d_kernel_size=4,   # минимальный валидный размер depthwise-свёртки
                                     # (это обычный nn.Conv1d, компиляция CUDA не нужна)
            qkv_proj_blocksize=4,
            num_heads=4,
        )
    ),
    context_length=MAX_LEN,
    num_blocks=4,
    embedding_dim=EMBED_DIM,
    slstm_at=[],  # используем только mLSTM-блоки (без sLSTM) — проще и стабильнее
)

class XLSTMScorer(nn.Module):
    def __init__(self, vocab_size, embed_dim, xlstm_config, dropout=0.1):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.xlstm_stack = xLSTMBlockStack(xlstm_config)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim // 2, 1),
        )

    def forward(self, input_ids, attention_mask):
        x = self.token_embedding(input_ids)          # (B, L, embed_dim)
        x = self.xlstm_stack(x)                        # (B, L, embed_dim)

        mask = attention_mask.unsqueeze(-1).float()     # (B, L, 1)
        summed = (x * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1)
        pooled = summed / counts                        # (B, embed_dim)

        score = self.head(pooled).squeeze(-1)
        return score


model = XLSTMScorer(VOCAB_SIZE, EMBED_DIM, xlstm_cfg).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"Всего параметров (обучается с нуля): {n_params/1e6:.2f}M")

Всего параметров (обучается с нуля): 9.51M


In [10]:
EPOCHS = 15  # больше, чем у RoBERTa/Mamba — модель учится с нуля, без претрейна,
              # нужно больше эпох, чтобы embedding-слой и xLSTM-блоки сошлись
LR = 1e-3     # выше, чем у fine-tuning (2e-5) — типичный LR для обучения с нуля

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.05 * total_steps), num_training_steps=total_steps)
criterion = nn.MSELoss()

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss = 0.0

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        target = batch["target"].to(DEVICE)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            pred = model(input_ids, attention_mask)
            loss = criterion(pred, target)

        if train:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        total_loss += loss.item() * len(target)

    return total_loss / len(loader.dataset)

In [11]:
best_val_loss = float("inf")
best_state = None

for epoch in range(EPOCHS):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}/{EPOCHS} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
torch.save(model.state_dict(), "xlstm_scorer.pt")
print("Лучший val_loss:", best_val_loss)

Epoch 1/15 | train_loss=0.27654 | val_loss=0.25734
Epoch 2/15 | train_loss=0.24374 | val_loss=0.25677
Epoch 3/15 | train_loss=0.22934 | val_loss=0.25060
Epoch 4/15 | train_loss=0.21701 | val_loss=0.26136
Epoch 5/15 | train_loss=0.20626 | val_loss=0.26281
Epoch 6/15 | train_loss=0.19701 | val_loss=0.26125
Epoch 7/15 | train_loss=0.18866 | val_loss=0.26998
Epoch 8/15 | train_loss=0.18107 | val_loss=0.26194
Epoch 9/15 | train_loss=0.17428 | val_loss=0.28203
Epoch 10/15 | train_loss=0.16742 | val_loss=0.27630
Epoch 11/15 | train_loss=0.16158 | val_loss=0.27325
Epoch 12/15 | train_loss=0.15577 | val_loss=0.28055
Epoch 13/15 | train_loss=0.15037 | val_loss=0.28116
Epoch 14/15 | train_loss=0.14544 | val_loss=0.28685
Epoch 15/15 | train_loss=0.14063 | val_loss=0.28894
Лучший val_loss: 0.25060206921845424


In [12]:
@torch.no_grad()
def predict_scores(df, loader):
    model.eval()
    preds = []
    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        pred = model(input_ids, attention_mask)
        preds.extend(pred.cpu().numpy().tolist())
    out_df = df.copy()
    out_df["xlstm_score"] = preds
    return out_df

test_df_with_preds = predict_scores(test_df, test_loader)
test_df_with_preds.to_csv("test_predictions_xlstm.csv", index=False)

In [13]:
results = {}
results["xlstm"] = evaluate_selection(test_df_with_preds, "xlstm_score")

oracle_res = evaluate_selection(test_df_with_preds, "cider")
results["oracle"] = oracle_res

rand_df = test_df_with_preds.copy()
rand_df["random_score"] = np.random.RandomState(SEED).rand(len(rand_df))
results["random"] = evaluate_selection(rand_df, "random_score")

comparison_df = pd.DataFrame(results).T
comparison_df = comparison_df[["avg_cider_selected", "top1_accuracy", "ndcg@3"]]
comparison_df = comparison_df.sort_values("avg_cider_selected", ascending=False)
print(comparison_df)

print()
print("Для сравнения: cross-encoder (RoBERTa, fine-tuned) = 0.871")
print("                bi-encoder (MiniLM frozen + MLP)    = 0.861")

        avg_cider_selected  top1_accuracy    ndcg@3
oracle            1.053640       1.000000  1.000000
xlstm             0.858863       0.338326  0.878064
random            0.761399       0.200870  0.798362

Для сравнения: cross-encoder (RoBERTa, fine-tuned) = 0.871
                bi-encoder (MiniLM frozen + MLP)    = 0.861
